# 실습 5 · K-평균 직접 만들기

**인공지능 일반**

## 0. 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 그래프에 한글을 쓰기 위한 준비 (실패해도 실습은 진행됩니다) ---
try:
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum > /dev/null 2>&1
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 설정 실패 — 그래프의 한글이 □□로 보일 수 있습니다.')

plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터

In [ ]:
rng = np.random.default_rng(0)
X = np.vstack([
    rng.normal([0, 0], 0.7, size=(100, 2)),
    rng.normal([5, 1], 0.7, size=(100, 2)),
    rng.normal([2, 5], 0.7, size=(100, 2)),
])
N, d = X.shape
K = 3

plt.scatter(X[:, 0], X[:, 1], s=12, c='gray')
plt.gca().set_aspect('equal'); plt.show()

## 2. Initialize — 센트로이드를 무작위로 정한다

In [ ]:
def initialize(X, K, seed=0):
    rng = np.random.default_rng(seed)
    low, high = X.min(axis=0), X.max(axis=0)
    centroid = np.zeros((K, d))
    for k in range(K):
        # ✏️ 실습 ① : low ~ high 사이의 무작위 값   힌트: rng.uniform(low, high)
        centroid[k, 0:d] = rng.uniform(low, high)
    return centroid

## 3. Step a — 각 점을 가장 가까운 센트로이드에 배정한다

In [ ]:
def assign(X, centroid):
    r = np.zeros((N, K), dtype=int)
    for i in range(N):
        dist = [np.sum((X[i] - centroid[j]) ** 2) for j in range(K)]
        for k in range(K):
            # ✏️ 실습 ② : k = argmin_j ‖x_i − μ_j‖² 이면 1, 아니면 0   힌트: np.argmin(dist)
            if k == np.argmin(dist):
                r[i, k] = 1
            else:
                r[i, k] = 0
    return r

## 4. Step b — 센트로이드를 배정된 점들의 평균으로 옮긴다

In [ ]:
def update(X, r, centroid):
    new_centroid = centroid.copy()
    for k in range(K):
        # ✏️ 실습 ③ : Σ_i r[i, k]·x_i  /  Σ_i r[i, k]   힌트: np.sum(r[:, k][:, None] * X, axis=0)
        numer = np.sum(r[:, k][:, None] * X, axis=0)
        denom = np.sum(r[:, k])
        if denom > 0:                    # 점이 없는 그룹은 그대로 둔다
            new_centroid[k] = numer / denom
    return new_centroid

## 5. Repeat until convergence — 소속이 바뀌지 않을 때까지 반복한다

In [ ]:
centroid = initialize(X, K, seed=0)
r = assign(X, centroid)
while True:
    centroid = update(X, r, centroid)
    new_r = assign(X, centroid)
    if np.array_equal(new_r, r):
        break
    r = new_r

plt.scatter(X[:, 0], X[:, 1], c=np.argmax(r, axis=1), cmap='tab10', vmin=0, vmax=9, s=12)
plt.scatter(centroid[:, 0], centroid[:, 1], c='k', marker='X', s=150)
plt.gca().set_aspect('equal'); plt.show()